# psycopg2 vs SQLAlchemy, Real Tradeoffs

**Module 4: Python Data Analysis**

**Learning Objectives:**
- Explain what SQLAlchemy's abstraction layer sits on top of, and what that layer actually buys you
- Compare `psycopg2` and SQLAlchemy on pooling, portability, error messages, and learning curve, using running code as the evidence
- Recommend one library for a given script scenario and defend the choice in writing

**Skills:** abstraction layer, connection pool, dialect, ORM


## Scenario

You are still on the city agency's data team, and the connection code from last class works. Now a second ask lands: a script that runs every ten minutes, all day, against the same `service_requests` table. Last class you proved both libraries return the same DataFrame. Today you find out whether that means the choice between them does not matter, by measuring what each one actually does to the database server.


## Before You Start

A script opens a database connection inside a loop that runs 500 times. The code is correct and it closes every single one. Is anything still wrong?

*Hint: Think about what the database server has to set up each time a brand-new connection arrives, not about what your Python code does.*


## Documentation

- Every Trip Through the Loop Opens a **New Connection**: https://www.postgresql.org/docs/current/functions-info.html

- One `engine`, and the **Same Connection** Comes Back: https://docs.sqlalchemy.org/en/20/core/pooling.html

- The Same Query, a **Different Database**: https://docs.sqlalchemy.org/en/20/core/engines.html


## Every Trip Through the Loop Opens a **New Connection**

*Context: `pg_backend_pid()` is PostgreSQL's own way of reporting which server process is handling your connection right now.*

*Complete the TODOs below as you code along.*

*Input:*


In [ ]:
import os
import psycopg2

url = os.environ["DATABASE_URL"]
for i in range(3):
    # TODO: open a connection, then get a cursor
    conn = ...
    cur = ...
    cur.execute("SELECT pg_backend_pid()")
    print("server process:", cur.fetchone()[0])
    # TODO: release the cursor, then the connection
    ...
    ...


## One `engine`, and the **Same Connection** Comes Back

*Complete the TODOs below as you code along.*

*Input:*


In [ ]:
from sqlalchemy import create_engine, text

# TODO: build one engine from the same url
engine = ...
q = text("SELECT pg_backend_pid()")

for i in range(3):
    with engine.connect() as conn:
        # TODO: run q and print the single value back
        print("server process:", ...)

print(engine.pool.status())


## The Same Query, a **Different Database**

*Context: `sqlite://` with nothing after it means a throwaway database that lives in memory and disappears when the notebook closes.*

*Fill in the blank (`____`) below.*

*Input:*


In [ ]:
import pandas as pd

sql = "SELECT borough, status FROM service_requests"
with engine.connect() as conn:
    pg_df = pd.read_sql(sql, conn)

lite = create_engine("____")
pg_df.to_sql("service_requests", lite, index=False)
lite_df = pd.read_sql(sql, lite)

print(lite.dialect.name, pg_df.equals(lite_df))
